In [1]:

import numpy as np
import pandas as pd
import random

def evolutionary_search_convergence(seed=42):

    np.random.seed(seed)
    random.seed(seed)

    G = 20
    population_size = 30
    elite_size = 10
    mutation_prob = 0.2
    crossover_prob = 0.8

    search_space = {
        "depth": [2, 3, 4, 5],
        "width": [16, 32, 64, 128],
        "kernel": [3, 5],
        "sparsity": [0.0, 0.25, 0.5],
        "precision": [8, 16, 32]
    }


    def create_architecture():
        return {
            "depth": random.choice(search_space["depth"]),
            "width": random.choice(search_space["width"]),
            "kernel": random.choice(search_space["kernel"]),
            "sparsity": random.choice(search_space["sparsity"]),
            "precision": random.choice(search_space["precision"])
        }


    def estimate_carbon(arch):
        complexity = (
            arch["depth"]
            * arch["width"]
            * arch["kernel"]
            * (arch["precision"] / 8)
            * (1 - arch["sparsity"])
        )

        carbon_cost = complexity / 1000

        return carbon_cost


    def estimate_accuracy(arch):
        accuracy = (
            0.55
            + 0.04 * arch["depth"]
            + 0.0015 * arch["width"]
            + 0.02 * (arch["kernel"] == 5)
            - 0.05 * arch["sparsity"]
            + 0.02 * (arch["precision"] == 16)
            + 0.03 * (arch["precision"] == 32)
        )

        accuracy += np.random.normal(0, 0.01)

        return min(accuracy, 0.90)


    def is_feasible(arch):
        carbon_budget = 0.85
        return estimate_carbon(arch) <= carbon_budget



    def fitness(arch):
        acc = estimate_accuracy(arch)
        carbon = estimate_carbon(arch)

        carbon_score = 1 - carbon

        if not is_feasible(arch):
            return 0.0

        final_score = (
            0.7 * acc
            + 0.3 * carbon_score
        )

        return round(final_score, 3)


    def mutate(arch):
        new_arch = arch.copy()

        if random.random() < mutation_prob:
            key = random.choice(list(search_space.keys()))
            new_arch[key] = random.choice(search_space[key])

        return new_arch



    def crossover(parent1, parent2):
        child = {}

        for key in search_space.keys():
            if random.random() < 0.5:
                child[key] = parent1[key]
            else:
                child[key] = parent2[key]

        return child



    population = []

    while len(population) < population_size:
        arch = create_architecture()

        if is_feasible(arch):
            population.append(arch)


    best_fitness_history = []
    avg_fitness_history = []
    infeasible_retained_history = []

    for generation in range(1, G + 1):

        scored_population = []

        for arch in population:
            score = fitness(arch)
            scored_population.append((arch, score))

        scored_population = sorted(
            scored_population,
            key=lambda x: x[1],
            reverse=True
        )

        best_score = scored_population[0][1]
        avg_score = np.mean([score for _, score in scored_population])

        best_fitness_history.append(round(best_score, 3))
        avg_fitness_history.append(round(avg_score, 3))
        infeasible_retained_history.append(0)

        elites = [
            arch for arch, score in scored_population[:elite_size]
        ]

        new_population = elites.copy()

        while len(new_population) < population_size:

            parent1 = random.choice(elites)
            parent2 = random.choice(elites)

            if random.random() < crossover_prob:
                child = crossover(parent1, parent2)
            else:
                child = parent1.copy()

            child = mutate(child)

            if is_feasible(child):
                new_population.append(child)

        population = new_population

    results = pd.DataFrame({
        "Generation": list(range(1, G + 1)),
        "Best Fitness": best_fitness_history,
        "Average Fitness": avg_fitness_history,
        "Infeasible Offspring Retained": infeasible_retained_history
    })



    print("Evolutionary Search Convergence Analysis")
    print("=" * 70)

    print(
        f"Initial Best Fitness: {best_fitness_history[0]}"
    )

    print(
        f"Best Fitness at Generation 14: "
        f"{best_fitness_history[13]}"
    )

    print(
        f"Final Best Fitness: {best_fitness_history[-1]}"
    )

    print(
        f"Infeasible Offspring Retained: "
        f"{sum(infeasible_retained_history)}"
    )

    print("\nGeneration-wise Results")
    print("=" * 70)

    print(results.to_string(index=False))

    return results


evolutionary_results = evolutionary_search_convergence()

Evolutionary Search Convergence Analysis
Initial Best Fitness: 0.771
Best Fitness at Generation 14: 0.801
Final Best Fitness: 0.804
Infeasible Offspring Retained: 0

Generation-wise Results
 Generation  Best Fitness  Average Fitness  Infeasible Offspring Retained
          1         0.771            0.693                              0
          2         0.793            0.744                              0
          3         0.794            0.769                              0
          4         0.806            0.781                              0
          5         0.804            0.787                              0
          6         0.807            0.790                              0
          7         0.815            0.783                              0
          8         0.805            0.789                              0
          9         0.803            0.786                              0
         10         0.803            0.783                            